In [ ]:
## Initialize

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from matplotlib import colors
import matplotlib.patches as patches
from scipy.spatial.distance import pdist, squareform
import pickle

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["text.usetex"] = True

norm_dir = Path.cwd().parent / "normalized"

df = pd.DataFrame(
    data=[
        ("'Round Midnight", r'RM\textsuperscript{*}', "Miles Davis", "1957", 0.0, 30.0, norm_dir / "02f518ed06fec892f39a52afedc1c256df217b60.wav"),
        ("This Is The Thing", "TITT", "Fink", "2007", 60.0, 30.0, norm_dir / "bfbff653d76ad4b891348a8b5c95f764e6b24332.wav"),
        ("Laser Beam", "LB", "Low", "2001", 0.0, 30.0, norm_dir / "d3ad985cc73bb6d3a5c3c303a2f9beaa56a1554a.wav"),
        ("Manifesto", "M", "Chilly Gonzales", "2025", 60.0, 30.0, norm_dir / "171826adeaf25374dcd3c4533b79cdd64a7a8e89.wav"),
        ("Cut And Clicks", "CAC", "Tetsu Inoue", "2000", 0.0, 30.0, norm_dir / "a91a691476e0ccb5cc746cdc402ceb8a5d5e0ad7.wav"),
        ("Fly Me To The Moon", "FMTTM", "Frank Sinatra", "2008?", 60.0, 30.0, norm_dir / "a5f3ee42bdf1397ac30b6f8e5ddd46dfb4f94757.wav"),
        ("I Saw Her Standing There", r'ISHST\textsuperscript{**}', "The Beatles", "1963", 0.0, 30.0, norm_dir / "af4451ce246b070f66fed280e53419e8bea4e578.wav"),
    ],
    columns=["title", "short_title", "artist", "year", "start_time", "duration", "audio_file"]
)

df

In [ ]:
# Load precomputed data

all_results = pickle.load(open("all_results.pickle", 'rb'))
print(all_results.keys())

In [ ]:
# Make ouput directories

Path('sip_images').mkdir(exist_ok=True)
Path('poster_images').mkdir(exist_ok=True)

In [ ]:
# Manipulate raw data

def asf_feats(coch, direction=True):
    return np.hstack([
        coch["asf"]["global_panning"], 
        coch["asf"]["panning_variation_accross_time"], 
        coch["asf"]["mean_over_time"] if direction else [],
        coch["asf"]["absolute_mean_over_time"], 
        coch["asf"]["std_over_time"]
])

def cepstrum(hist, L=20):
    log_hist = np.log(hist)
    Em = np.append(log_hist, log_hist[::-1])
    cepstrum_coeffs = np.fft.ifft(Em)
    return np.real(cepstrum_coeffs[:L])

    
spf_features = np.stack([track["vector"] for track in all_results["spf_results"]])
ph_gomez_features = np.stack([track["vector"] for track in all_results["ph_gomez_results"]])
ph_sarroff_features = np.stack([track["vector"] for track in all_results["ph_sarroff_results"]])
ph_cepstrum = np.apply_along_axis(cepstrum, 1, ph_gomez_features)
asf_features = np.stack([asf_feats(coch) for coch in all_results["asf_results"]])

print(f"{spf_features.shape=}")
print(f"{ph_gomez_features.shape=}")
print(f"{ph_sarroff_features.shape=}")
print(f"{ph_cepstrum.shape=}")
print(f"{asf_features.shape=}")

In [ ]:
def norm_arr(arr):
    arr_min = np.min(arr)
    arr_max = np.max(arr)
    # Avoid division by zero
    if arr_max == arr_min:
        return np.zeros_like(arr)
    return (arr - arr_min) / (arr_max - arr_min)

def draw_matrix(matrix, ax, axis_title, titles, norm=True, cmap='Greys'):
    """
    Displays the lower triangle of a symmetric matrix on a given matplotlib axis,
    with cell values (formatted to 2 decimals) in high-contrast text.
    Args:
        matrix (2D list or np.ndarray): The square matrix to display.
        ax (matplotlib.axes.Axes): The axis to draw the matrix on.
        cmap (str): Colormap for the image. Default is 'Blues'.
    """
    if norm:
        matrix = norm_arr(np.array(matrix))
    n = matrix.shape[0]
    # mask = np.triu(np.ones_like(matrix, dtype=bool), k=1)
    # masked_matrix = np.ma.array(matrix, mask=mask)

    # mat_mean = np.mean(masked_matrix)
    # mat_std = np.std(masked_matrix)
    # print(axis_title, mat_mean-mat_std, mat_mean+mat_std)

    norm = colors.Normalize(vmin=matrix.min(), vmax=matrix.max())
    # img = ax.imshow(masked_matrix, cmap=cmap, aspect='equal', norm=norm, interpolation='nearest')

    cmap_obj = plt.get_cmap(cmap)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if i >= j:
                value = matrix[i, j]
                # Get normalized value for this cell to access the color
                normed_val = norm(value)
                rgba = cmap_obj(normed_val)
                # Calculate luminance (perceived brightness)
                r, g, b = rgba[:3]
                luminance = 0.299*r + 0.587*g + 0.114*b
                text_color = 'black' if luminance > 0.5 else 'white'
                # Draw rectangle
                linewidth, edgecolor = (0.5, 'gray')
                rect = patches.Rectangle(
                    (j - 0.5, n - i - 1.5), 1, 1, 
                    fill=True,
                    facecolor=rgba,
                    linestyle='-',
                    edgecolor=edgecolor,
                    linewidth=linewidth,
                )
                ax.add_patch(rect)
                # Write cell value
                ax.text(
                    j, n-i-1, f'{value:.2f}',
                    ha='center', va='center',
                    color=text_color, fontsize=12
                )
    
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_yticks(range(matrix.shape[0]))
    ax.set_xticklabels(titles, rotation=270)
    ax.set_yticklabels(reversed(titles))
    ax.set_xlim(-0.5, n-0.4)
    ax.set_ylim(-0.5, n-0.4)

    # Remove the top and right axes
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_title(axis_title)

    # return img

spf_dist = pdist(spf_features, 'euclidean')
ph_dist = pdist(ph_sarroff_features, 'euclidean')
phc_dist = pdist(ph_cepstrum, 'euclidean')
asf_dist = pdist(asf_features, 'euclidean')

# Draw diagram for paper - 4 matrices in a row
fig, ax = plt.subplots(1, 4, figsize=(16, 4), layout='constrained')
norm = True
draw_matrix(squareform(ph_dist), ax[0], "Panning Histogram", df["short_title"], norm=norm)
draw_matrix(squareform(phc_dist), ax[1], "Panning Coefficients", df["short_title"], norm=norm)
draw_matrix(squareform(asf_dist), ax[2], "Amplitude Stereo Features", df["short_title"], norm=norm)
draw_matrix(squareform(spf_dist), ax[3], "Stereo Panning Features", df["short_title"], norm=norm)
fig.savefig("sip_images/distance_matrices.pdf")


# Draw diagram for poster - matrices in a 2x2 grid with biases circled
fig, ax = plt.subplots(2, 2, figsize=(8, 8), layout='tight')
draw_matrix(squareform(ph_dist), ax[0,0], "Panning Histogram", df["short_title"], norm=norm)
draw_matrix(squareform(phc_dist), ax[0,1], "Panning Coefficients", df["short_title"], norm=norm)
draw_matrix(squareform(asf_dist), ax[1,0], "Amplitude Stereo Features", df["short_title"], norm=norm)
draw_matrix(squareform(spf_dist), ax[1,1], "Stereo Panning Features", df["short_title"], norm=norm)
fig.savefig("poster_images/distance_matrices.pdf")

In [ ]:
from sklearn.decomposition import PCA


spf_2d = PCA(2).fit_transform(spf_features)
ph_2d = PCA(2).fit_transform(ph_sarroff_features)
phc_2d = PCA(2).fit_transform(ph_cepstrum)
asf_2d = PCA(2).fit_transform(asf_features)
fig, ax = plt.subplots(2, 2, figsize=(8, 8), layout='tight')
colours = list("rkkkkkb")
ax[0,0].scatter(spf_2d[:,0], spf_2d[:,1], color=colours)
ax[0,1].scatter(ph_2d[:,0], ph_2d[:,1], color=colours)
ax[1,0].scatter(phc_2d[:,0], phc_2d[:,1], color=colours)
ax[1,1].scatter(asf_2d[:,0], asf_2d[:,1], color=colours)

def label_points(ax, p, txt):
    for i, txt in enumerate(txt):
        ax.annotate(txt, p[i,:])

label_points(ax[0,0], spf_2d, df["short_title"])
label_points(ax[0,1], ph_2d, df["short_title"])
label_points(ax[1,0], phc_2d, df["short_title"])
label_points(ax[1,1], asf_2d, df["short_title"])

In [ ]:
Dm = []
Ds = []
B = []

bias_table_dists = [ph_dist, phc_dist, asf_dist, spf_dist]

for dists in bias_table_dists:
    mat = norm_arr(squareform(dists))
    distance_from_mono = np.mean(mat[0, 1:-1])
    distance_from_stereo = np.mean(mat[-1, 1:-1])
    ratio = distance_from_mono / distance_from_stereo
    log_ratio = np.log(ratio)
    Dm.append(distance_from_mono)
    Ds.append(distance_from_stereo)
    B.append(log_ratio)

bias_table = pd.DataFrame({
    "$D_m$": Dm,
    "$D_s$": Ds,
    "$B$": B,
},
index=["PH", "PC", "ASF", "SPF"],
)
display(bias_table)
print(bias_table.style.format(precision=2).to_latex())

In [ ]:
def norm_upper(arr):
    return arr / np.max(arr)

def sarroff_hists_vertical():
    fig, ax = plt.subplots(3,1, figsize=(4,5), layout="constrained", sharex=True)
    X = np.arange(-64,65)
    ax[0].plot(X, ph_sarroff_features[0], color='k', label="'Round Midnight", linewidth=0.8)
    ax[1].plot(X, ph_sarroff_features[4], color='k', label="Cut And Clicks", linewidth=0.8)
    ax[2].plot(X, ph_sarroff_features[6], color='k', label="I Saw Her Standing There", linewidth=0.8)
    ts = df["short_title"][[0,4,6]].to_list()
    bottoms = [-0.04, -0.04, -0.001]

    for a,t,btm in zip(ax, ts, bottoms):
        a.set_xticks([-64, 0, 64], labels=["L", "C", "R"])
        a.set_xticks([-64, -48, -32, -16, 0, 16, 32, 48, 64], minor=True)
        a.set_ylabel("Histogram value")
        a.text(0.98, 0.94, t, horizontalalignment='right', verticalalignment='top', transform=a.transAxes)
        a.set_ylim(bottom=btm)

    ax[-1].set_xlabel("Azimuth")
    return fig


def sarroff_hists_horizontal():
    fig, ax = plt.subplots(1,3, figsize=(4,2.5), layout="constrained")
    X = np.arange(-64,65)
    ax[0].plot(X, ph_sarroff_features[0], color='k', label="'Round Midnight", linewidth=0.8)
    ax[1].plot(X, ph_sarroff_features[4], color='k', label="Cut And Clicks", linewidth=0.8)
    ax[2].plot(X, ph_sarroff_features[6], color='k', label="I Saw Her Standing There", linewidth=0.8)
    ts = df["short_title"][[0,4,6]].to_list()
    bottoms = [-0.04, -0.04, -0.001]

    for a,t,btm in zip(ax, ts, bottoms):
        a.set_xticks([-64, 0, 64], labels=["L", "C", "R"])
        a.set_xticks([-64, -48, -32, -16, 0, 16, 32, 48, 64], minor=True)
        a.set_yticks([])
        a.text(0.98, 0.94, t, horizontalalignment='right', verticalalignment='top', transform=a.transAxes)
        a.set_ylim(bottom=btm)
        a.set_xlabel("Azimuth")

    ax[0].set_ylabel("Histogram value")
    return fig

def sarroff_hists_overlay():    
    fig, ax = plt.subplots(1,1, figsize=(4,2.5), layout="constrained")
    X = np.arange(-64,65)
    ts = df["short_title"][[0,4,6]].to_list()
    ax.plot(X, norm_upper(ph_sarroff_features[0]), color='k', linewidth=0.8, linestyle='-', label=ts[0])
    ax.plot(X, norm_upper(ph_sarroff_features[4]), color='k', linewidth=0.8, linestyle='--', label=ts[1])
    ax.plot(X, norm_upper(ph_sarroff_features[6]), color='k', linewidth=0.8, linestyle=':', label=ts[2])

    ax.set_xticks([-64, 0, 64], labels=["L", "C", "R"])
    ax.set_xticks([-64, -48, -32, -16, 0, 16, 32, 48, 64], minor=True)
    ax.set_yticks([])
    ax.set_xlabel("Azimuth")

    ax.set_ylabel("Normalized histogram value")
    ax.legend()
    return fig

def sarroff_hists_overlay2():
    fig, ax = plt.subplots(1,1, figsize=(4,2.5), layout="constrained")
    X = np.arange(-64,65)
    ts = df["short_title"][[0,4,6]].to_list()
    ax.plot(X, ph_sarroff_features[6], color='k', linewidth=0.8, linestyle=':', label=ts[2])
    ax2 = ax.twinx()
    ax2.plot(X, ph_sarroff_features[0], color='k', linewidth=0.8, linestyle='-', label=ts[0])
    ax2.plot(X, ph_sarroff_features[4], color='k', linewidth=0.8, linestyle='--', label=ts[1])

    ax.set_xticks([-64, 0, 64], labels=["L", "C", "R"])
    ax.set_xticks([-64, -48, -32, -16, 0, 16, 32, 48, 64], minor=True)
    # ax.set_yticks([])
    ax.set_xlabel("Azimuth")

    ax.set_ylabel(r"Histogram value, ISHST\textsuperscript{**}")
    ax2.set_ylabel(r"Histogram value, RM\textsuperscript{*}/CAC")
    fig.legend()
    return fig

def sarroff_hists_mono_stereo():
    fig, ax = plt.subplots(1,2, figsize=(4,2.5), layout="constrained")
    X = np.arange(-64,65)
    ts = df["short_title"][[0,4,6]].to_list()
    ax[0].plot(X, ph_sarroff_features[0], color='k', linewidth=0.8, linestyle='-', label=ts[0])
    ax[0].plot(X, ph_sarroff_features[4], color='k', linewidth=0.8, linestyle='--', label=ts[1])
    ax[1].plot(X, ph_sarroff_features[6], color='k', linewidth=0.8, linestyle='-', label=ts[2])

    for i in range(2):
        ax[i].set_xticks([-64, 0, 64], labels=["L", "C", "R"])
        ax[i].set_xticks([-64, -48, -32, -16, 0, 16, 32, 48, 64], minor=True)
        ax[i].set_xlabel("Azimuth")

    ax[0].set_ylim(top=0.6)
    ax[0].set_ylabel("Histogram value")
    ax[0].legend()
    ax[1].legend()

    return fig

def sarroff_hists_mono_stereo_vert():
    fig, ax = plt.subplots(2,1, figsize=(4, 3.4), layout="constrained", sharex=True)
    X = np.arange(-64,65)
    ts = df["short_title"][[0,4,6]].to_list()
    ax[0].plot(X, ph_sarroff_features[0], color='k', linewidth=0.8, linestyle='-', label=ts[0])
    ax[0].plot(X, ph_sarroff_features[4], color='k', linewidth=0.8, linestyle='--', label=ts[1])
    ax[1].plot(X, ph_sarroff_features[6], color='k', linewidth=0.8, linestyle='-', label=ts[2])

    for i in range(2):
        ax[i].set_xticks([-64, 0, 64], labels=["L", "C", "R"])
        ax[i].set_xticks([-64, -48, -32, -16, 0, 16, 32, 48, 64], minor=True)
        ax[i].set_ylabel("Histogram value")
        ax[i].yaxis.set_major_formatter("{x:.02f}")

    # ax[0].set_ylim(top=0.7)
    # ax[1].set_ylim(top=0.044)
    ax[1].set_xlabel("Azimuth")
    ax[0].legend()
    ax[1].legend()

    return fig

fig = sarroff_hists_mono_stereo_vert()
fig.savefig("sip_images/ph_rm_cac_ishst_overlay.pdf")


def sarroff_hist_single(ax, ph, short_title):
    # fig, ax = plt.subplots(1,1, figsize=(4, 3.4), layout="constrained")
    X = np.arange(-64,65)
    # ts = df["short_title"][i]
    # tt = df["title"][i]
    ax.plot(X, ph, color='k', linewidth=0.8, linestyle='-', label=short_title)
    ax.text(0.98,0.96, short_title, horizontalalignment='right', verticalalignment='top', transform=ax.transAxes)
    # ax.legend()

    for i in range(2):
        ax.set_xticks([-64, 0, 64], labels=["L", "C", "R"])
        ax.set_xticks([-64, -48, -32, -16, 0, 16, 32, 48, 64], minor=True)
        ax.set_ylabel("Histogram value")
        ax.yaxis.set_major_formatter("{x:.02f}")


fig, axes = plt.subplots(7,1, figsize=(4, 21), layout="constrained", sharex=True)
axes[-1].set_xlabel("Azimuth")
for ax, ph, st in zip(axes, ph_sarroff_features, df["short_title"]):
    sarroff_hist_single(ax, ph, st)
fig.savefig(f"poster_images/panning_histogram - all.pdf")

In [ ]:
class KiloNoDecimalFormatter:
    """
    Tick formatter for kilo-units
    """

    @staticmethod
    def _kilo_formatter(x: float, pos: float) -> str:
        if x < 1000:
            return str(int(x))
        else:
            return f"{x / 1000:.0f}k"

    def formatter(self) -> plt.FuncFormatter:
        return plt.FuncFormatter(self._kilo_formatter)

fig, axes = plt.subplots(2,1, figsize=(4, 3.2), layout='constrained', sharex=True)
hop_size = 512
fft_size = 1024
fs = 44100
freqs = np.fft.rfftfreq(fft_size, d=1/fs)[1:] # remove DC
t = np.arange(all_results["ph_gomez_results"][4]["panning_spectrogram"].shape[0]) * hop_size / fs

f1, f2 = 1200, 1800
t1, t2 = t[[f1, f2]]

for ax, track_idx in zip(axes, [4,6]):
    img = ax.imshow(
        np.rot90(all_results["ph_gomez_results"][track_idx]["panning_spectrogram"][f1:f2]), \
        extent=(t1, t2, 0, freqs[-1]),
        vmax=1,
        vmin=0,
        aspect='auto',
        interpolation='none',
        cmap='gray',
    )
    ax.yaxis.set_major_formatter(KiloNoDecimalFormatter().formatter())
    ax.set_ylabel("Frequency (Hz)")
    ax.text(0.96, 
            0.93, 
            df["short_title"].loc[track_idx],
            backgroundcolor='white',
            horizontalalignment='right', 
            verticalalignment='top', 
            transform=ax.transAxes
        )
    # ax.set_xlabel("Time (s)")
    cbar = fig.colorbar(img, ax=ax, ticks=[0, 0.5, 1])
    cbar.ax.set_yticklabels(['L', 'C', 'R'])
    
axes[1].set_xlabel("Time (s)")
fig.savefig("sip_images/sps_cac_ishst.pdf")

In [ ]:
asf_different_lufs = np.array([asf_feats(coch) for coch in all_results["cochleagrams_different_lufs"]])
print(asf_different_lufs.shape)
asf_different_lufs_dists = pdist(asf_different_lufs)

asf_different_lufs_titles = [
    r"RM",
    r"ISHST-23", 
    r"ISHST-26", 
    r"ISHST-29", 
]

fig, ax = plt.subplots(1,1, figsize=(2.5,2.5), layout='constrained')
draw_matrix(squareform(asf_different_lufs_dists), ax, "", asf_different_lufs_titles)

fig.savefig("sip_images/distance_matrix_different_lufs.pdf")

In [ ]:
def draw_coch_pair(coch, title, ax, xlim):
    ax[0].errorbar(
        coch[0]["asf"]["mean_over_time"],
        coch[0]["subbands"],
        xerr=coch[0]["asf"]["std_over_time"],
        color="black",
        fmt=" .",
        linewidth=0.4,
    )
    ax[0].set_yscale('log')
    ax[0].set_ylim((20,15000))
    ax[0].set_xlim(xlim)
    ax[0].set_ylabel("Frequency (Hz)")
    ax[0].set_xlabel("Panning")
    ax[0].text(0.96,0.97, title[0], horizontalalignment='right', verticalalignment='top', transform=ax[0].transAxes)

    ax[1].errorbar(
        coch[1]["asf"]["mean_over_time"],
        coch[1]["subbands"],
        xerr=coch[1]["asf"]["std_over_time"],
        color="black",
        fmt=" .",
        linewidth=0.4,
    )
    ax[1].set_xlim(xlim)
    ax[1].set_xlabel("Panning")
    ax[1].text(0.96,0.97, title[1], horizontalalignment='right', verticalalignment='top', transform=ax[1].transAxes)

fig, ax = plt.subplots(1,2, figsize=(4,2.5), layout='constrained', sharey=True)
draw_coch_pair([all_results["asf_results"][1], all_results["asf_results"][4]], df['short_title'][[1,4]].to_list(), ax, (-2,2))
fig.savefig("sip_images/asf_titt_cac.pdf")


fig, ax = plt.subplots(1,2, figsize=(4,2.5), layout='constrained', sharey=True)
draw_coch_pair([all_results["asf_results"][5], all_results["asf_results"][6]], df['short_title'][[5,6]].to_list(), ax, (-6,6))
fig.savefig("sip_images/asf_fmttm_ishst.pdf")


def draw_coch(coch, title, ax, xlim):
    ax.errorbar(
        coch["asf"]["mean_over_time"],
        coch["subbands"],
        xerr=coch["asf"]["std_over_time"],
        color="black",
        fmt=" .",
        linewidth=0.4,
    )
    ax.set_yscale('log')
    ax.set_ylim((20,15000))
    ax.set_xlim(xlim)
    # ax.set_ylabel("Frequency (Hz)")
    # ax.set_xlabel("Panning")
    ax.text(0.98,0.97, title, horizontalalignment='right', verticalalignment='top', transform=ax.transAxes)

def asf_supplementary():
    fig, ax = plt.subplots(7,1, figsize=(4,21), layout='constrained', sharex=True)
    for i in range(7):
        draw_coch(all_results["asf_results"][i], df['short_title'][i], ax[i], (-7,7))
        ax[i].set_ylabel("Frequency (Hz)")
    ax[-1].set_xlabel("Panning")
    fig.savefig("poster_images/amplitude_stereo_features - all.pdf")

asf_supplementary()

In [ ]:
import matplotlib as mpl
t = mpl.markers.MarkerStyle(marker='d', fillstyle='none')
t._transform = t.get_transform().rotate_deg(90)

def draw_spf(spf, ax, title):
    mm = spf[0:4]
    ms = spf[8:12]
    sm = spf[4:8]
    ss = spf[12:16]
    X1 = np.linspace(0,4,4)
    X2 = np.linspace(4.8,8,4)
    X2 = X1 + 0.4
    X3 = np.mean([X1, X2], axis=0)
    ax.errorbar(
        X1, 
        mm, 
        yerr=ms, 
        linestyle='none',
        fillstyle="none",
        marker=t,
        color='k', 
        label='Long-term',
        linewidth=0.618,
    )
    ax.set_ylim(-0.05,0.8)
    ax.set_xlim(-0.5, 5)
    ax.text(0.96,0.97, title, horizontalalignment='right', verticalalignment='top', transform=ax.transAxes)
    ax.errorbar(
        X2, 
        sm, 
        yerr=ss, 
        linestyle='none',
        fillstyle="none",
        marker='.',
        color='k', 
        label='Short-term',
        linewidth=0.618,
    )
    ax.set_xticks(X3, ["Fullband", "Lows", "Mids", "Highs"], rotation=90)

fig, ax = plt.subplots(
    1,3, 
    figsize=(4,2.5), 
    layout='constrained', 
    sharey=True
)

draw_spf(spf_features[0], ax[0], df["short_title"][0])
draw_spf(spf_features[4], ax[1], df["short_title"][4])
draw_spf(spf_features[6], ax[2], df["short_title"][6])
ax[0].set_ylabel("Panning")

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.24, 1.14), ncols=2)
fig.savefig("sip_images/spf_rm_cac_ishst.pdf")


# supplementary
fig, ax = plt.subplots(
    1,7, 
    figsize=(10,2.5), 
    layout='constrained', 
    sharey=True
)
ax[0].set_ylabel("Panning")
for i in range(7):
    draw_spf(spf_features[i], ax[i], df["short_title"][i])
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.39, 1.14), ncols=2)
fig.savefig("poster_images/stereo_panning_features - all.pdf")

In [ ]:
def cepstrum_reverse(cepstrum_coeffs):
    regenerated_log_spect = np.real(np.fft.fft(cepstrum_coeffs)) # un-IDFT 
    regenerated_Em = np.exp(regenerated_log_spect) # un-log
    regenerated_hist = regenerated_Em[:len(regenerated_Em)//2] # un-mirror
    regenerated_hist = regenerated_hist / (sum(regenerated_hist) * len(regenerated_hist))  # normalize
    return regenerated_hist

def paper():
    fig, axes = plt.subplots(2,2, figsize=(4,4), layout="constrained", sharey=True, sharex=True)
    for a, ph, title in zip(axes.flatten(), ph_gomez_features[[2,3,5,6]], df["short_title"][[2,3,5,6]]):
        quant = cepstrum_reverse(cepstrum(ph))
        X_bar = np.linspace(-0.9, 0.9, len(quant))
        bar_width = (X_bar[1] - X_bar[0])
        X_line = np.linspace(-1, 1, len(ph))
        a.bar(X_bar, quant, width=bar_width, color='white', edgecolor='black', label='quantized')
        a.set_xticks([-1, 0, 1], labels='LCR')
        a.plot(X_line, ph, color='k', linewidth=0.8, label='original')
        a.set_ylim(bottom=0, top=0.03)
        a.text(0.96, 0.96, title, horizontalalignment='right', verticalalignment='top', transform=a.transAxes)
    axes[0,0].set_ylabel("Histogram value")
    axes[1,0].set_ylabel("Histogram value")
    axes[1,0].set_xlabel("Azimuth")
    axes[1,1].set_xlabel("Azimuth")
    handles, labels = axes[0,0].get_legend_handles_labels()
    fig.legend(labels=labels, handles=handles, bbox_to_anchor=(0.34, 0.6, 0.5, 0.5), ncol=2)
    fig.savefig("sip_images/phc_quant_4.pdf")


def plot_phc(ax, ph, title):
    quant = cepstrum_reverse(cepstrum(ph))
    X_bar = np.linspace(-0.9, 0.9, len(quant))
    bar_width = (X_bar[1] - X_bar[0])
    X_line = np.linspace(-1, 1, len(ph))
    ax.bar(X_bar, quant, width=bar_width, color='white', edgecolor='black', label='quantized')
    ax.plot(X_line, ph, color='k', linewidth=0.8, label='original')
    ax.set_xticks([-1, 0, 1], labels='LCR')
    ax.set_ylabel("Histogram value")
    ax.text(0.98,0.97, title, horizontalalignment='right', verticalalignment='top', transform=ax.transAxes)
    if title == "FMTTM":
        ax.set_ylim(top=0.015)

def supplementary():
    fig, axes = plt.subplots(7,1, figsize=(4, 21), layout="constrained", sharex=True)
    axes[-1].set_xlabel("Azimuth")
    for ax, ph, title in zip(axes, ph_gomez_features, df["short_title"]):
        plot_phc(ax, ph, title)
    fig.savefig("poster_images/panning_coefficients - all.pdf")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(labels=labels, handles=handles, bbox_to_anchor=(0.38, 0.6, 0.5, 0.42), ncol=2)

supplementary()

In [ ]:
# Poster QR code


import segno

qrcode = segno.make_qr("https://github.com/richardmitic/dafx2025")
qrcode.save(
    "poster_images/qr.pdf",
    dark="black",
)
